# YOLO26n signal classification v1

Train a 3-class YOLO26n classifier for `left-arrow`, `right-arrow`, and `stop-signal` from the Kaggle dataset `jeffreyamc/yolo-signals-input`.

## Setup and progress tracking

Install current Ultralytics support, define constants, and keep `run_summary.json` updated.

In [ ]:
import json
import random
import shutil
import subprocess
import sys
import traceback
from pathlib import Path

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "ultralytics"])

from ultralytics import YOLO

SEED = 42
CLASSES = ["left-arrow", "right-arrow", "stop-signal"]
DATASET_SLUG = "yolo-signals-input"
MODEL_NAME = "yolo26n-cls.pt"
WORKING = Path("/kaggle/working")
PREPARED = WORKING / "yolo_signals_cls"
RUNS = WORKING / "runs"
SUMMARY_PATH = WORKING / "run_summary.json"
METRICS_PATH = WORKING / "metrics.json"

progress = {
    "status": "running",
    "stage": "setup",
    "dataset": "jeffreyamc/yolo-signals-input",
    "model": MODEL_NAME,
    "classes": CLASSES,
    "artifacts": [],
}

def save_progress():
    SUMMARY_PATH.write_text(json.dumps(progress, indent=2), encoding="utf-8")

def record_artifact(path):
    path = Path(path)
    if path.exists():
        rel = str(path.relative_to(WORKING)) if path.is_relative_to(WORKING) else str(path)
        if rel not in progress["artifacts"]:
            progress["artifacts"].append(rel)

save_progress()
print("Setup complete")

## Resolve and validate dataset

Find the mounted Kaggle dataset and count images per class.

In [ ]:
try:
    progress["stage"] = "resolve_dataset"
    save_progress()

    candidates = [
        Path("/kaggle/input") / DATASET_SLUG,
        Path("/kaggle/input/datasets/jeffreyamc") / DATASET_SLUG,
    ]
    candidates.extend([p for p in Path("/kaggle/input").glob("**/left-arrow") if p.is_dir()])

    data_root = None
    for candidate in candidates:
        root = candidate.parent if candidate.name == "left-arrow" else candidate
        if all((root / name).is_dir() for name in CLASSES):
            data_root = root
            break

    if data_root is None:
        raise FileNotFoundError("Could not find dataset folders: " + ", ".join(CLASSES))

    counts = {name: len(list((data_root / name).glob("*.jpg"))) for name in CLASSES}
    if any(count == 0 for count in counts.values()):
        raise ValueError(f"Empty class folder found: {counts}")

    progress["data_root"] = str(data_root)
    progress["source_counts"] = counts
    save_progress()
    print("Dataset root:", data_root)
    print("Counts:", counts)
except Exception as exc:
    progress["status"] = "failed"
    progress["stage"] = "resolve_dataset"
    progress["error"] = repr(exc)
    progress["traceback"] = traceback.format_exc()
    save_progress()
    raise

## Prepare train and validation split

Create an 80/20 folder split in Ultralytics classification format.

In [ ]:
try:
    progress["stage"] = "prepare_split"
    save_progress()

    if PREPARED.exists():
        shutil.rmtree(PREPARED)

    split_counts = {"train": {}, "val": {}}
    random.seed(SEED)

    for class_name in CLASSES:
        images = sorted((data_root / class_name).glob("*.jpg"))
        random.shuffle(images)
        val_count = max(1, round(len(images) * 0.2))
        val_set = set(images[:val_count])

        for image_path in images:
            split = "val" if image_path in val_set else "train"
            out_dir = PREPARED / split / class_name
            out_dir.mkdir(parents=True, exist_ok=True)
            shutil.copy2(image_path, out_dir / image_path.name)

        split_counts["train"][class_name] = len(images) - val_count
        split_counts["val"][class_name] = val_count

    progress["prepared_root"] = str(PREPARED)
    progress["split_counts"] = split_counts
    save_progress()
    print("Prepared:", PREPARED)
    print("Split counts:", split_counts)
except Exception as exc:
    progress["status"] = "failed"
    progress["stage"] = "prepare_split"
    progress["error"] = repr(exc)
    progress["traceback"] = traceback.format_exc()
    save_progress()
    raise

## Train YOLO26n classifier

Fine-tune `yolo26n-cls.pt`, validate the best checkpoint, and publish compact outputs.

In [ ]:
try:
    progress["stage"] = "train"
    save_progress()

    model = YOLO(MODEL_NAME)
    results = model.train(
        data=str(PREPARED),
        epochs=80,
        imgsz=224,
        batch=32,
        patience=12,
        device=0,
        seed=SEED,
        workers=2,
        cache=True,
        project=str(RUNS),
        name="yolo26n_cls_v1",
        exist_ok=True,
        verbose=True,
    )

    save_dir = Path(getattr(model.trainer, "save_dir", getattr(results, "save_dir", RUNS / "yolo26n_cls_v1")))
    weights_dir = save_dir / "weights"
    best_src = weights_dir / "best.pt"
    last_src = weights_dir / "last.pt"
    if not best_src.exists():
        raise FileNotFoundError(f"Missing trained checkpoint: {best_src}")

    best_out = WORKING / "best.pt"
    last_out = WORKING / "last.pt"
    shutil.copy2(best_src, best_out)
    record_artifact(best_out)
    if last_src.exists():
        shutil.copy2(last_src, last_out)
        record_artifact(last_out)

    progress["stage"] = "validate"
    save_progress()

    trained = YOLO(str(best_out))
    metrics = trained.val(data=str(PREPARED), imgsz=224, device=0, verbose=False)
    metric_payload = {
        "top1": float(getattr(metrics, "top1", 0.0)),
        "top5": float(getattr(metrics, "top5", 0.0)),
        "fitness": float(getattr(metrics, "fitness", 0.0)) if not callable(getattr(metrics, "fitness", None)) else float(metrics.fitness()),
        "save_dir": str(save_dir),
        "best_checkpoint": "best.pt",
        "last_checkpoint": "last.pt" if last_out.exists() else None,
        "split_counts": progress["split_counts"],
    }
    METRICS_PATH.write_text(json.dumps(metric_payload, indent=2), encoding="utf-8")
    record_artifact(METRICS_PATH)

    for pattern in ["*.png", "*.jpg", "*.csv", "args.yaml", "results.csv"]:
        for src in save_dir.glob(pattern):
            dst = WORKING / src.name
            shutil.copy2(src, dst)
            record_artifact(dst)

    progress["metrics"] = metric_payload
    progress["status"] = "complete"
    progress["stage"] = "done"
    save_progress()
    record_artifact(SUMMARY_PATH)
    print(json.dumps(progress, indent=2))
except Exception as exc:
    progress["status"] = "failed"
    progress["error"] = repr(exc)
    progress["traceback"] = traceback.format_exc()
    save_progress()
    print(json.dumps(progress, indent=2))